# Phase 4 — Train & Forecast (XGBoost)

This notebook is the **last stage** of the pipeline:

```
HLS imagery  →  Prithvi embeddings  →  Feature fusion (+ weather)  →  master_features.parquet
                                                                              │
                                                          NASS yields  ──────►│
                                                                              ▼
                                                           **THIS NOTEBOOK** ── XGBoost LOOCV
                                                                              │
                                                                              ▼
                                                              2025 yield forecast + analog cone
```

**Inputs it expects** (produced by `feature_fusion.combine_all_states`):
- `data/processed/master_features.parquet` — columns `[year, month, county, fips, state, embedding_vector]` with 786-dim vectors (768 Prithvi + 18 weather).
- `data/raw/corn_yield_2005_2024.csv` — NASS county yields.

**Status check**: HLS feature extraction is still running, so `master_features.parquet` does not exist yet. The notebook detects this and falls back to a small **synthetic** dataset so you can validate the training logic now and re-run as-is once fusion completes.

In [ ]:
# Cell 1 — Setup & paths
import os, sys
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR       = PROJECT_ROOT / "data" / "raw"

MASTER_FEATURES = PROCESSED_DIR / "master_features.parquet"
YIELD_CSV       = RAW_DIR / "corn_yield_2005_2024.csv"

print("Project root           :", PROJECT_ROOT)
print("Master features exists :", MASTER_FEATURES.exists())
print("Yield CSV exists       :", YIELD_CSV.exists())

## Cell 2 — Load features + yields (with synthetic fallback)

If `master_features.parquet` isn't on disk yet (feature fusion still running), we build a small synthetic dataset with the same schema so the training pipeline below runs end-to-end. **Any results from synthetic data are meaningless — they only prove the code works.**

In [ ]:
# Cell 2 — Load or synthesize
EMBED_DIM = 786          # 768 Prithvi + 18 weather
USE_SYNTHETIC = not MASTER_FEATURES.exists()

if not USE_SYNTHETIC:
    feature_df = pd.read_parquet(MASTER_FEATURES)
    print(f"Loaded real features: {len(feature_df)} rows, "
          f"embedding dim = {feature_df['embedding_vector'].iloc[0].shape}")
else:
    print("master_features.parquet not found — generating synthetic stand-in.")
    rng = np.random.default_rng(42)
    counties = [
        ("19", "STORY",      "IA"),
        ("19", "BLACK HAWK", "IA"),
        ("31", "YORK",       "NE"),
        ("55", "SAUK",       "WI"),
        ("29", "HOWARD",     "MO"),
        ("08", "YUMA",       "CO"),
    ]
    rows = []
    for year in range(2015, 2026):
        for fips, county, state in counties:
            for month in [5, 6, 7, 8, 9, 10]:
                rows.append({
                    "year": year,
                    "month": month,
                    "county": county,
                    "fips": fips,
                    "state": state,
                    "embedding_vector": rng.standard_normal(EMBED_DIM).astype(np.float32),
                })
    feature_df = pd.DataFrame(rows)

# --- yields ---
if YIELD_CSV.exists():
    yield_df = pd.read_csv(YIELD_CSV)
else:
    print("Yield CSV missing — synthesizing yields too.")
    rng2 = np.random.default_rng(7)
    yr_rows = []
    for year in feature_df["year"].unique():
        if year >= 2025:
            continue  # we don't have 2025 truth
        for (fips, county, state), _ in feature_df.groupby(["fips", "county", "state"]):
            yr_rows.append({
                "year": int(year),
                "state_fips": fips,
                "county_name": county,
                "yield_bu_acre": float(rng2.normal(170, 15)),
            })
    yield_df = pd.DataFrame(yr_rows)

# Normalize yield join keys
yield_df["state_fips"]  = yield_df["state_fips"].astype(str).str.zfill(2)
yield_df["county_name"] = yield_df["county_name"].astype(str).str.upper()

print(f"\nfeature_df: {len(feature_df)} rows")
print(f"yield_df  : {len(yield_df)} rows")
feature_df.head(3)

## Cell 3 — Build the seasonal signature per (year, county)

For each `(year, county)` we average the embedding across the months observable **before** the forecast date. This mirrors `forecaster.get_analog_years` so training and inference see identical features.

In [ ]:
# Cell 3 — Seasonal signatures
from forecaster import DATE_MAP   # {'aug1': [5,6,7], 'sep1':[...], 'oct1':[...], 'final':[5..10]}

FORECAST_DATE = "final"   # try 'aug1' / 'sep1' / 'oct1' for earlier-season models
months = DATE_MAP[FORECAST_DATE]
print(f"Knowledge window for '{FORECAST_DATE}': months {months}")

f = feature_df.copy()
f = f[f["month"].isin(months)]

# canonicalize join keys to match yield_df
if "county_name" not in f.columns:
    f = f.rename(columns={"county": "county_name"})
if "state_fips" not in f.columns:
    f["state_fips"] = f["fips"].astype(str).str.zfill(5).str[:2]
f["state_fips"]  = f["state_fips"].astype(str).str.zfill(2)
f["county_name"] = f["county_name"].astype(str).str.upper()

seasonal = (
    f.groupby(["year", "state_fips", "county_name"])["embedding_vector"]
     .apply(lambda g: np.mean(np.stack(g.values), axis=0))
     .reset_index()
)
print(f"Signatures: {len(seasonal)} (year, county) rows")
seasonal.head(3)

## Cell 4 — Train with Time-Series Split (XGBoost)

We use the helpers in `src/train_model.py`. LOOCV is appropriate because we have ~10–20 years per county — every (year, county) row gets held out once.

In [ ]:
# Cell 4 — Train + LOOCV
from train_model import prepare_matrices, train_and_validate

# train_model.prepare_matrices expects merge keys: ['year', 'state_fips', 'county']
sig_for_train = seasonal.rename(columns={"county_name": "county"})
yld_for_train = yield_df.rename(columns={"county_name": "county"})

X, y = prepare_matrices(sig_for_train, yld_for_train)
print(f"X shape: {X.shape}   y shape: {y.shape}")

mae, actuals, preds = train_and_validate(X, y)

## Cell 5 — Diagnostics: predicted vs. actual + residuals

In [ ]:
# Cell 5 — Plots
import matplotlib.pyplot as plt

actuals = np.asarray(actuals)
preds   = np.asarray(preds)
resid   = preds - actuals

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(actuals, preds, alpha=0.6)
lo = float(min(actuals.min(), preds.min()))
hi = float(max(actuals.max(), preds.max()))
axes[0].plot([lo, hi], [lo, hi], "k--", lw=1)
axes[0].set_xlabel("Actual yield (bu/acre)")
axes[0].set_ylabel("Predicted yield (bu/acre)")
axes[0].set_title(f"LOOCV predictions  (MAE = {mae:.2f})")

axes[1].hist(resid, bins=20, edgecolor="black")
axes[1].axvline(0, color="red", lw=1)
axes[1].set_xlabel("Residual (pred − actual)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Residuals  (mean = {resid.mean():+.2f}, std = {resid.std():.2f})")

plt.tight_layout()
plt.show()

if USE_SYNTHETIC:
    print("\n⚠ Synthetic data — metrics are meaningless. Re-run after feature fusion completes.")

## Cell 6 — Final fit on all years + 2025 forecast

After validation, retrain on the full history and combine with the analog-year cone from `forecaster.get_analog_years` to produce both a **point estimate** and an **uncertainty range**. This cell only does work when 2025 partial-season features are present.

In [ ]:
# Cell 6 — Final fit + 2025 forecast (runs only when 2025 features are present)
import xgboost as xgb
from forecaster import get_analog_years

QUERY_YEAR = 2025
has_query_year = (seasonal["year"] == QUERY_YEAR).any()

if not has_query_year:
    print(f"No {QUERY_YEAR} signatures yet — skipping forecast step.")
else:
    final_model = xgb.XGBRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.07,
        subsample=0.8, objective="reg:squarederror", random_state=42,
    )
    final_model.fit(X, y)
    print("Final model trained on all historical years.")

    query_rows = sig_for_train[sig_for_train["year"] == QUERY_YEAR]
    forecasts = []
    for _, r in query_rows.iterrows():
        x_q = r["embedding_vector"].reshape(1, -1)
        y_hat = float(final_model.predict(x_q)[0])

        analog = get_analog_years(
            state_fips=r["state_fips"],
            county_name=r["county"],
            forecast_date=FORECAST_DATE,
            feature_df=feature_df,
            yield_df=yield_df,
            query_year=QUERY_YEAR,
            top_k=5,
        )
        if isinstance(analog, pd.DataFrame) and not analog.empty:
            cone_lo = float(analog["yield_bu_acre"].min())
            cone_hi = float(analog["yield_bu_acre"].max())
        else:
            cone_lo = cone_hi = float("nan")

        forecasts.append({
            "state_fips": r["state_fips"],
            "county": r["county"],
            "xgb_point_estimate": y_hat,
            "analog_cone_low":   cone_lo,
            "analog_cone_high":  cone_hi,
        })

    forecast_df = pd.DataFrame(forecasts)
    display(forecast_df)

## Where to extend next

- **Hyperparameter tuning** — once real features are loaded, sweep `max_depth`, `learning_rate`, `n_estimators` with `GridSearchCV` (still LOO-style on `year`).
- **Feature importance** — `final_model.get_booster().get_score(importance_type='gain')` to see whether the weather (first 18 dims) or Prithvi (last 768 dims) carries more signal.
- **Per-state models** — if the 5-state pooled model under-performs, train one XGBoost per state.
- **Earlier forecast dates** — change `FORECAST_DATE` to `aug1`/`sep1`/`oct1` and compare MAE; this is the headline "skill vs. lead-time" plot for the demo.